# Логарифмирование признаков и целевой переменной

Когда, зачем и как применять логарифмическое преобразование в задачах машинного обучения.

In [ ]:
!uv pip install pandas scikit-learn seaborn statsmodels > /dev/null 2>&1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import FunctionTransformer, PowerTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline

sns.set_style("whitegrid")
rng = np.random.RandomState(42)
pd.options.display.float_format = "{:.4f}".format

---
## 1. Зачем нужно логарифмирование

Логарифмирование — одно из самых часто используемых преобразований в ML. Оно нужно в трёх основных случаях:

| Ситуация | Что делаем | Зачем |
|---|---|---|
| Признак имеет **сильно скошенное распределение** (правосторонняя асимметрия) | $\log(x)$ или $\log(1+x)$ | Многие модели (линейная регрессия, нейросети) лучше работают с симметричными распределениями |
| **Целевая переменная** скошена, residuals показывают гетероскедастичность («рупор») | $\log(y)$ | Стабилизирует дисперсию, исправляет гетероскедастичность |
| Связь между признаком и целевой переменной — **нелинейная**, например степенная $y \sim x^a$ | $\log(x)$ или $\log(y)$ | Превращает нелинейную связь в линейную |

### Важно: какие модели выигрывают от логарифмирования?

- **Линейные модели** (LinearRegression, LogisticRegression, Ridge, Lasso, SVM) — ДА, очень полезно
- **Нейросети** — ДА, помогает стабилизировать обучение
- **kNN, методы на расстояниях** — ДА
- **Деревья решений, RandomForest, GradientBoosting** — **НЕТ**, деревья не чувствительны к монотонным преобразованиям признаков

> **Правило**: если модель основана на расстояниях или на линейных комбинациях — логарифмирование скошенных признаков помогает. Если модель основана на деревьях — не помогает (но и не вредит).

---
## 2. Логарифмирование признаков

### 2.1. Визуализация эффекта

Посмотрим, как логарифмирование меняет распределение признака.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

# --- Логнормальное распределение ---
x_lognorm = rng.lognormal(mean=0, sigma=1.0, size=2000)

axes[0,0].hist(x_lognorm, bins=60, color='steelblue', edgecolor='white')
axes[0,0].set_title('Логнормальное распределение (цены, доходы)')
axes[1,0].hist(np.log(x_lognorm), bins=60, color='coral', edgecolor='white')
axes[1,0].set_title('После log(x)')

# --- Экспоненциальное распределение ---
x_exp = rng.exponential(scale=2.0, size=2000)

axes[0,1].hist(x_exp, bins=60, color='steelblue', edgecolor='white')
axes[0,1].set_title('Экспоненциальное распределение')
axes[1,1].hist(np.log(x_exp + 1e-6), bins=60, color='coral', edgecolor='white')
axes[1,1].set_title('После log(x)')

# --- Степенное распределение (Pareto) ---
x_pareto = (np.random.pareto(a=1.5, size=2000) + 1) * 10

axes[0,2].hist(x_pareto, bins=60, color='steelblue', edgecolor='white')
axes[0,2].set_title('Степенной закон (частоты слов, трафик)')
axes[1,2].hist(np.log(x_pareto), bins=60, color='coral', edgecolor='white')
axes[1,2].set_title('После log(x)')

plt.tight_layout()
plt.show()

### 2.2. Пример: как логарифмирование признака улучшает линейную регрессию

Сгенерируем данные, где целевая переменная зависит от признака степенным образом:

$$y = a \cdot x^b + \varepsilon$$

Линейная регрессия без преобразования будет плохо предсказывать такие данные. Но если мы возьмём $\log(x)$ — зависимость станет линейной.

In [ ]:
# Генерируем данные с нелинейной зависимостью
n = 500
x = np.linspace(0.1, 10, n)                         # признак, избегаем нулей для log
y = 2.0 * x ** 0.6 + rng.normal(0, 0.5, size=n)     # y ~ x^0.6 + шум

X = x.reshape(-1, 1)

# Модель БЕЗ логарифмирования
lr = LinearRegression()
lr.fit(X, y)
y_pred = lr.predict(X)

# Модель С логарифмированием признака
X_log = np.log(X)                                    # log(x) как новый признак
lr_log = LinearRegression()
lr_log.fit(X_log, y)
y_pred_log = lr_log.predict(X_log)

# --- Визуализация ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Исходные данные
axes[0].scatter(x, y, alpha=0.3, s=10)
axes[0].plot(x, y_pred, 'r-', lw=2, label='Линейная регрессия (без log)')
axes[0].plot(x, y_pred_log, 'g-', lw=2, label='Линейная регрессия (c log(x))')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Данные и предсказания')
axes[0].legend()

# 2. Зависимость в логарифмическом масштабе
axes[1].scatter(np.log(x), y, alpha=0.3, s=10, color='gray')
axes[1].scatter(np.log(x), y_pred_log, alpha=0.3, s=10, color='green')
axes[1].set_xlabel('log(x)')
axes[1].set_ylabel('y')
axes[1].set_title('Зависимость после log(x)')

# 3. Сравнение метрик
r2_1 = r2_score(y, y_pred)
r2_2 = r2_score(y, y_pred_log)
mse_1 = mean_squared_error(y, y_pred)
mse_2 = mean_squared_error(y, y_pred_log)

metrics = pd.DataFrame({
    'Модель': ['Без log(x)', 'С log(x)'],
    'R2': [f'{r2_1:.3f}', f'{r2_2:.3f}'],
    'MSE': [f'{mse_1:.3f}', f'{mse_2:.3f}']
})
axes[2].axis('off')
table = axes[2].table(cellText=metrics.values,
                       colLabels=metrics.columns,
                       cellLoc='left', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2)
axes[2].set_title('Сравнение метрик', fontsize=12)

plt.tight_layout()
plt.show()

print(f"R2 без логарифмирования: {r2_1:.3f}")
print(f"R2 с логарифмированием:  {r2_2:.3f}")
print()
print("Коэффициент при x в модели без log:", lr.coef_[0])
print("Коэффициент при log(x) в модели с log:", lr_log.coef_[0])

#### Что произошло?

Зависимость $y \sim x^{0.6}$ — нелинейна. Линейная регрессия по $x$ с этим не справляется. Но после $\log(x)$ зависимость становится линейной:

$$y \approx 2 \cdot x^{0.6} \;\Longrightarrow\; \log(y) \approx \log(2) + 0.6 \cdot \log(x)$$

Здесь мы не трогали $y$, только преобразовали $x$. В следующем разделе увидим, как работает $\log(y)$.

---
## 3. Логарифмирование целевой переменной

Логарифмирование $y$ нужно, когда:
1. **Целевая переменная сильно скошена** (например, цены, доходы, счётчики)
2. **Гетероскедастичность** — разброс ошибок растёт с ростом предсказанного значения (график остатков — «рупор»)
3. **Мультипликативные ошибки** — ошибка пропорциональна величине, а не аддитивна

### 3.1. Пример с гетероскедастичностью

Создадим данные, где разброс $y$ растёт пропорционально $x$:

In [ ]:
# Генерируем данные с гетероскедастичностью
n = 500
x = np.linspace(1, 10, n)

# Чем больше x, тем больше разброс y
y_true = 3 + 0.8 * x
y = y_true + rng.normal(0, scale=0.3 * x, size=n)

X = x.reshape(-1, 1)

# --- Модель без логарифмирования y ---
lr = LinearRegression()
lr.fit(X, y)
y_pred = lr.predict(X)
residuals = y - y_pred

# --- Модель с логарифмированием y ---
lr_log = LinearRegression()
lr_log.fit(X, np.log(y))
y_pred_log = np.exp(lr_log.predict(X))
residuals_log = y - y_pred_log

# --- Визуализация ---
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# 1. Данные и предсказания
axes[0,0].scatter(x, y, alpha=0.3, s=10)
axes[0,0].plot(x, y_pred, 'r-', lw=2, label='Без log(y)')
axes[0,0].plot(x, y_pred_log, 'g-', lw=2, label='С log(y)')
axes[0,0].set_title('Данные и предсказания')
axes[0,0].legend()

# 2. Распределение остатков (без log)
axes[0,1].scatter(y_pred, residuals, alpha=0.4, color='red')
axes[0,1].axhline(0, color='black', ls='--')
axes[0,1].set_title('Остатки: без log(y) — виден «рупор»')
axes[0,1].set_xlabel('Предсказанное y')
axes[0,1].set_ylabel('Остатки')

# 3. Распределение остатков (с log)
axes[0,2].scatter(y_pred_log, residuals_log, alpha=0.4, color='green')
axes[0,2].axhline(0, color='black', ls='--')
axes[0,2].set_title('Остатки: с log(y) — гомоскедастичность')
axes[0,2].set_xlabel('Предсказанное y')
axes[0,2].set_ylabel('Остатки')

# 4-5. Распределение целевой переменной
axes[1,0].hist(y, bins=50, color='steelblue', edgecolor='white')
axes[1,0].set_title('Распределение y')
axes[1,1].hist(np.log(y), bins=50, color='coral', edgecolor='white')
axes[1,1].set_title('Распределение log(y)')

# 6. Метрики (в исходной шкале)
axes[1,2].axis('off')
metrics = pd.DataFrame({
    'Модель': ['Без log(y)', 'С log(y)'],
    'R2': [f'{r2_score(y, y_pred):.3f}', f'{r2_score(y, y_pred_log):.3f}'],
    'MAE': [f'{mean_absolute_error(y, y_pred):.3f}', f'{mean_absolute_error(y, y_pred_log):.3f}']
})
table = axes[1,2].table(cellText=metrics.values,
                        colLabels=metrics.columns,
                        cellLoc='left', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2)
axes[1,2].set_title('Метрики в исходной шкале y', fontsize=11)

plt.tight_layout()
plt.show()

#### Что мы видим?

**Без $\log(y)$:** Остатки растут с ростом предсказанного значения — классический «рупор» (гетероскедастичность). Это нарушает предположения линейной регрессии и может приводить к неточным оценкам.

**С $\log(y)$:** Остатки распределены равномерно — модель «одинаково ошибается» и для маленьких, и для больших значений.

> **Интуиция**: ошибка в 500 тыс. руб. для квартиры за 5 млн и за 50 млн — это разные вещи. Логарифмирование делает ошибки относительными (в процентах), а не абсолютными.

### 3.2. Правильный способ: `TransformedTargetRegressor`

В scikit-learn есть специальный класс `TransformedTargetRegressor`, который:
- Автоматически применяет $\log(1+y)$ к целевой переменной при обучении
- Автоматически применяет обратное преобразование $\exp(y)-1$ к предсказаниям
- Работает в пайплайнах и с кросс-валидацией

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Без логарифмирования
lr_plain = LinearRegression()
lr_plain.fit(X_train, y_train)
y_pred_plain = lr_plain.predict(X_test)

# С логарифмированием через TransformedTargetRegressor
lr_transformed = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,          # log(1 + y)
    inverse_func=np.expm1   # exp(y) - 1
)
lr_transformed.fit(X_train, y_train)
y_pred_trans = lr_transformed.predict(X_test)

# Сравнение
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, y_pred_plain, alpha=0.4, color='red')
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
axes[0].set_title(f'Без log(y):  R2 = {r2_score(y_test, y_pred_plain):.3f}')
axes[0].set_xlabel('Истинные значения')
axes[0].set_ylabel('Предсказания')

axes[1].scatter(y_test, y_pred_trans, alpha=0.4, color='green')
axes[1].plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
axes[1].set_title(f'С log(y) (TransformedTargetRegressor):  R2 = {r2_score(y_test, y_pred_trans):.3f}')
axes[1].set_xlabel('Истинные значения')
axes[1].set_ylabel('Предсказания')

plt.tight_layout()
plt.show()

print(f"MAE без log(y):  {mean_absolute_error(y_test, y_pred_plain):.3f}")
print(f"MAE с log(y):   {mean_absolute_error(y_test, y_pred_trans):.3f}")

---
## 4. Когда логарифмирование НЕ нужно

### 4.1. Деревья решений и ансамбли

Проверим: деревья инвариантны к монотонным преобразованиям, поэтому логарифмирование признаков им не помогает.

In [ ]:
# ---- Тот же пример с RandomForest ----
rf_plain = RandomForestRegressor(n_estimators=200, random_state=42)
rf_plain.fit(X_train, y_train)
y_pred_rf = rf_plain.predict(X_test)

rf_log = RandomForestRegressor(n_estimators=200, random_state=42)
rf_log.fit(np.log(X_train + 1), y_train)
y_pred_rf_log = rf_log.predict(np.log(X_test + 1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, y_pred_rf, alpha=0.4)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
axes[0].set_title(f'RandomForest без log(x):  R2 = {r2_score(y_test, y_pred_rf):.3f}')
axes[0].set_xlabel('Истинные значения')
axes[0].set_ylabel('Предсказания')

axes[1].scatter(y_test, y_pred_rf_log, alpha=0.4)
axes[1].plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
axes[1].set_title(f'RandomForest с log(x):  R2 = {r2_score(y_test, y_pred_rf_log):.3f}')
axes[1].set_xlabel('Истинные значения')
axes[1].set_ylabel('Предсказания')

plt.tight_layout()
plt.show()

print(f"Случайный лес без log(x):  R2 = {r2_score(y_test, y_pred_rf):.3f}")
print(f"Случайный лес с log(x):   R2 = {r2_score(y_test, y_pred_rf_log):.3f}")
print()
print("Результаты практически не отличаются — деревьям всё равно.")

## 5. Практические приёмы

### 5.1. Проблема нулей: log vs log1p

Если в данных есть нули, $\log(0) = -\infty$. Используйте $\log(1+x)$ (`np.log1p`):

In [ ]:
data_with_zeros = np.array([0, 0.1, 1, 10, 100, 1000])

print("Исходные данные:", data_with_zeros)
print("log(x):       ", np.log(data_with_zeros))       # -inf на первом элементе
print("log1p(x):     ", np.log1p(data_with_zeros))     # log(1+x) — безопасно

### 5.2. Автоматический подбор: PowerTransformer (Box-Cox)

Если не уверены, какое степенное преобразование применить — используйте `PowerTransformer` с методом Box-Cox. Он сам подберёт оптимальный параметр $\lambda$:

$$x^{(\lambda)} = \begin{cases} \frac{x^\lambda - 1}{\lambda}, & \lambda \neq 0 \\ \ln(x), & \lambda = 0 \end{cases}$$

In [ ]:
# Сравним ручное логарифмирование и Box-Cox
x_skewed = rng.lognormal(mean=0, sigma=1.5, size=2000)

boxcox = PowerTransformer(method='box-cox')
x_bc = boxcox.fit_transform(x_skewed.reshape(-1, 1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(x_skewed, bins=60, color='steelblue', edgecolor='white')
axes[0].set_title(f'Исходное распределение\nskew = {pd.Series(x_skewed).skew():.2f}')

axes[1].hist(np.log(x_skewed), bins=60, color='coral', edgecolor='white')
axes[1].set_title(f'После log(x)\nskew = {pd.Series(np.log(x_skewed)).skew():.2f}')

axes[2].hist(x_bc, bins=60, color='purple', edgecolor='white')
axes[2].set_title(f'Box-Cox (lambda = {boxcox.lambdas_[0]:.2f})\nskew = {pd.Series(x_bc.flatten()).skew():.2f}')

plt.tight_layout()
plt.show()

print(f"Подобранное lambda = {boxcox.lambdas_[0]:.3f}")
print(f"lambda = 0 соответствует log-преобразованию")

### 5.3. Обратное преобразование предсказаний

Если вы логарифмировали $y$, предсказания модели будут в логарифмической шкале. Их нужно преобразовать обратно:

| Прямое преобразование | Обратное | numpy |
|---|---|---|
| $\tilde{y} = \log(y)$ | $y = \exp(\tilde{y})$ | `np.exp()` |
| $\tilde{y} = \log(1+y)$ | $y = \exp(\tilde{y}) - 1$ | `np.expm1()` |

**Важная тонкость:** при обратном преобразовании возникает систематическое смещение. Если ошибки модели в логарифмической шкале нормальны с дисперсией $\sigma^2$, то несмещённая оценка:

$$\hat{y}_{\text{corrected}} = \exp(\tilde{y}) \cdot \exp(\sigma^2 / 2)$$

На практике это смещение часто игнорируют, либо просто используют `TransformedTargetRegressor`, который «не знает» о поправке.

### 5.4. Использование в пайплайне с кросс-валидацией

Лучший способ — объединить все преобразования в `Pipeline` и подбирать параметры через `GridSearchCV`:

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

# Пайплайн: log1p признака -> масштабирование -> Ridge-регрессия
pipeline = make_pipeline(
    FunctionTransformer(np.log1p, validate=True),
    Ridge(alpha=1.0, random_state=42)
)

# Сравним с обычной регрессией на тех же данных
# (используем данные из примера 2.2: y ~ x^0.6)
X_pipe = x.reshape(-1, 1)
lr_pipe = LinearRegression()

scores_plain = cross_val_score(lr_pipe, X_pipe, y, cv=5, scoring='r2')
scores_log = cross_val_score(pipeline, X_pipe, y, cv=5, scoring='r2')

print(f"R2 (LinearRegression):   {scores_plain.mean():.3f} +/- {scores_plain.std():.3f}")
print(f"R2 (log(x) -> Ridge):     {scores_log.mean():.3f} +/- {scores_log.std():.3f}")

---
## 6. Сводка: когда и что логарифмировать

| Ситуация | Что делать | Типичные примеры |
|---|---|---|
| Признак сильно скошен вправо | $\log(x)$, $\log(1+x)$ или Box-Cox | Цены, доходы, площадь, счётчики (число просмотров, число покупок) |
| Целевая переменная скошена | $\log(y)$ через `TransformedTargetRegressor` | Цены на жильё, зарплаты, страховые выплаты |
| Гетероскедастичность (рупор) | $\log(y)$ — часто исправляет | Любая регрессия с непостоянной дисперсией |
| Нелинейная связь $y \sim x^a$ | $\log(x)$ или $\log(y)$ (или оба) | Спрос от цены, площадь от радиуса |
| В данных есть нули | `np.log1p(x)` = $\log(1+x)$ | Разрежённые признаки, счётчики с нулями |
| Используете деревья (RF, GBM) | **Ничего не даёт** | Можно не логарифмировать |
| Нужен автоматический подбор | `PowerTransformer(method='box-cox')` | Когда не знаете, какое преобразование подойдёт |

### Связь с законом Вебера — Фехнера

Как уже упоминалось в ноутбуке [features.ipynb](features.ipynb), психофизический закон Вебера — Фехнера гласит, что **ощущение пропорционально логарифму стимула**. Это проявляется в ML: модель часто «воспринимает» разницу между 1 млн и 2 млн примерно так же, как между 10 млн и 20 млн — пропорционально, а не абсолютно. Логарифмирование как раз это и моделирует.